# AMRFinderPlus on the genomes on Google Drive

Finds the known resistance genes and mutations in every genome downloaded by `download_genomes_to_drive.ipynb`
(`My Drive/FYP/genomes_lab_v7/`), with the project's own `experiments/genome/features/run_amrfinder.py`.
One result per genome goes to `My Drive/FYP/amrfinder_lab_v7/<genome_id>.tsv`; `build_gene_matrix.py` later
joins them with the earlier 24,926 into the gene table.

- **Same tool and database as the earlier genomes:** AMRFinderPlus 4.2.7 with database 2026-08-07.1 (copied from
  Ali's Mac to Drive), so old and new results are comparable. Tested: gzipped genomes give results identical to the
  earlier run.
- **Mycobacterium is left out** (23,491 genomes): AMRFinderPlus has no tuberculosis mode, so it would find no
  resistance mutations there; the k-mers cover them. That leaves 41,359 genomes, 99% with organism-specific
  detection (point mutations).
- **Runtime:** Runtime → Change runtime type → CPU, **High-RAM** (8 cores). About 2.5 days in total, over several
  sessions; a few compute units per day.
- **Resumable:** finished genomes are skipped. If Colab stops: Runtime → Disconnect and delete runtime, then Run all.

## 1. Connect Google Drive

In [ ]:
# If this says "Mountpoint must not already contain files":
# Runtime → Disconnect and delete runtime, then Run all again.
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings

In [ ]:
import os
FOLDER = '/content/drive/MyDrive/FYP'
GENOMES = f'{FOLDER}/genomes_lab_v7'          # the downloaded .fna.gz files
LIST = f'{FOLDER}/lab_genomes_todo.csv'       # genome_id, taxon_id, genus
TAXA = f'{FOLDER}/code/taxon_species.csv'     # species of each taxon ID
OUT = f'{FOLDER}/amrfinder_lab_v7'            # one .tsv per genome
THREADS = 2                                   # per genome
JOBS = max(1, os.cpu_count() // THREADS)      # genomes at a time
LIMIT = ''                                    # e.g. '--limit 10' for a quick test
print(f'{os.cpu_count()} CPU cores: {JOBS} genomes at a time')

## 3. Install AMRFinderPlus 4.2.7 and the database (about 3 to 5 minutes, every session)

In [ ]:
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
!/content/bin/micromamba create -y -q -p /content/amrf -c conda-forge -c bioconda ncbi-amrfinderplus=4.2.7
!mkdir -p /content/amrfinder_db && tar -xzf "$FOLDER/code/amrfinder_db_2026-08-07.1.tar.gz" -C /content/amrfinder_db
os.environ['AMRFINDER_DB'] = '/content/amrfinder_db/2026-08-07.1'
!cp "$FOLDER/code/run_amrfinder.py" /content/run_amrfinder.py
!/content/amrf/bin/amrfinder --version
!/content/amrf/bin/amrfinder --database_version 2>&1 | grep 'Database version'

## 4. Run
Prints a line every 25 genomes with the time left. Leave it running; if it stops, run all cells again.

In [ ]:
!python /content/run_amrfinder.py --amrfinder /content/amrf/bin/amrfinder --genomes "$GENOMES" --list "$LIST" --taxa "$TAXA" --out "$OUT" --skip-genus Mycobacterium --jobs $JOBS --threads $THREADS $LIMIT

## 5. Check progress

In [ ]:
import glob
done = len(glob.glob(f'{OUT}/*.tsv'))
print(f'{done:,} of 41,359 genomes done ({100 * done / 41359:.1f}%)')

## 6. Make sure everything reached Drive (before closing the tab)

In [ ]:
drive.flush_and_unmount()
print('Done.')